# Person Face Events - project runner

This notebook is set up to be run manually on your machine.

It does not execute anything here. It only gives you a clean, root-safe entry point that:

- finds the project root automatically
- lists the project files
- checks the core Python packages and model files
- points you to the scripts and notebooks you should run next


In [ ]:
from pathlib import Path

def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "main.py").exists() and (candidate / "src").exists():
            return candidate
    raise FileNotFoundError("Could not find the project root from the current notebook location.")

ROOT = find_project_root()
print(f"Project root: {ROOT}")


In [ ]:
print("Project files:\n")
for path in sorted(ROOT.rglob("*")):
    if any(part in {".git", ".venv", "__pycache__"} for part in path.parts):
        continue
    if path.is_file():
        print(path.relative_to(ROOT))


In [ ]:
import sys

sys.path.insert(0, str(ROOT))

print("Python:", sys.executable)

for mod in ["ultralytics", "cv2", "yaml", "numpy", "onnxruntime", "insightface"]:
    try:
        module = __import__(mod)
        version = getattr(module, "__version__", "n/a")
        print(f"{mod}: {version}")
    except Exception as exc:
        print(f"{mod}: FAIL -> {exc}")


In [ ]:
from src.utils.config import load_settings

cfg = load_settings(ROOT / "config" / "settings.yaml")

print("Camera source:", cfg["camera"]["source"])
print("YOLO weights  :", cfg["models"]["yolo_weights"])
print("Face root     :", cfg["models"]["face_root"])
print("Events DB     :", cfg["events"]["db_path"])
print("Faces DB      :", cfg["events"]["faces_db_path"])
print("Gallery root  :", cfg["gallery"]["images_dir"])

for label, rel in [
    ("YOLO PT", cfg["models"]["yolo_weights"]),
    ("YOLO ONNX", cfg["models"].get("yolo_onnx")),
    ("Tracker config", cfg["models"]["tracker_config"]),
]:
    if rel:
        p = Path(rel)
        print(f"{label}: {'OK' if p.exists() else 'MISSING'} -> {p}")


## What to run next

1. `python scripts/enroll_face.py`
2. `python main.py`

If `ultralytics` appears installed but you still see no output, the common causes are:

- the notebook kernel is not using the project venv
- the camera/RTSP source is unavailable
- the YOLO weights are not where `config/settings.yaml` expects them
